# Amazon Customer Data Preparation & Data Modeling

This project focuses on preparing and transforming Amazon customer and sales data into structured datasets ready for analysis and SQL-based data exploration.

Using **Python and Pandas**, the project loads the cleaned Amazon dataset, performs data preparation and transformation, and organizes the data into separate dimension and fact tables following a **star-schema approach**.

### Key Components

* Customer dimension (`dim_customer`)
* Product dimension (`dim_product`)
* Time dimension (`dim_time`)
* Sales fact table (`fact_sale`)
* Data type conversion and preparation
* Data transformation using Pandas
* Export of analytical tables to CSV files

The resulting datasets provide a clean and structured foundation for further **SQL analysis, Excel reporting, and business intelligence**.


In [45]:
import pandas as pd

In [46]:
df = pd.read_csv("Amazon_cleaned.csv")

In [47]:
df_copy = df.copy()

In [48]:
df_copy.head(5)

,OrderID,OrderDate,OrderYear,OrderMonth,CustomerID,CustomerName,ProductID,ProductName,Category,Brand,...,GrossAmount,DiscountedAmount,NetAmount,TotalAmount,PaymentMethod,OrderStatus,City,State,Country,SellerID
0,ORD0000001,2023-01-31,2023,January,CUST001504,Vihaan Sharma,P00014,Drone Mini,Books,BrightLux,...,319.77,0.00,319.77,319.86,Debit Card,Delivered,Washington,DC,India,SELL01967
1,ORD0000002,2023-12-30,2023,December,CUST000178,Pooja Kumar,P00040,Microphone,Home & Kitchen,UrbanStyle,...,251.37,12.57,238.80,259.64,Amazon Pay,Delivered,Fort Worth,TX,United States,SELL01298
2,ORD0000003,2022-05-10,2022,May,CUST047516,Sneha Singh,P00044,Power Bank 20000mAh,Clothing,UrbanStyle,...,105.09,10.51,94.58,108.06,Debit Card,Delivered,Austin,TX,United States,SELL00908
3,ORD0000004,2023-07-18,2023,July,CUST030059,Vihaan Reddy,P00041,Webcam Full HD,Home & Kitchen,Zenith,...,167.90,25.18,142.72,159.66,Cash on Delivery,Delivered,Charlotte,NC,India,SELL01164
4,ORD0000005,2023-02-04,2023,February,CUST048677,Aditya Kapoor,P00029,T-Shirt,Clothing,KiddoFun,...,1031.28,257.82,773.46,821.36,Credit Card,Cancelled,San Antonio,TX,Canada,SELL01411


In [49]:
df_copy["OrderDate"]=pd.to_datetime(df_copy["OrderDate"])

### Create dim_customer

In [50]:
dim_customer = (df_copy[["CustomerID","CustomerName","City","State","Country"]]
                .drop_duplicates(subset="CustomerID")
                .reset_index(drop=True)
                )
dim_customer

,CustomerID,CustomerName,City,State,Country
0,CUST001504,Vihaan Sharma,Washington,DC,India
1,CUST000178,Pooja Kumar,Fort Worth,TX,United States
2,CUST047516,Sneha Singh,Austin,TX,United States
3,CUST030059,Vihaan Reddy,Charlotte,NC,India
4,CUST048677,Aditya Kapoor,San Antonio,TX,Canada
...,...,...,...,...,...
43228,CUST011350,Mohit Sharma,Seattle,WA,Canada
43229,CUST044331,Sahil Gupta,Columbus,OH,United States
43230,CUST011195,Aditya Kumar,Denver,CO,United States
43231,CUST005627,Kabir Reddy,San Antonio,TX,United States


### Create dim_products

In [51]:
dim_product = (df_copy[["ProductID","ProductName","Category","Brand"]]
               .drop_duplicates(subset="ProductID")
               .reset_index(drop=True)
               )
dim_product

,ProductID,ProductName,Category,Brand
0,P00014,Drone Mini,Books,BrightLux
1,P00040,Microphone,Home & Kitchen,UrbanStyle
2,P00044,Power Bank 20000mAh,Clothing,UrbanStyle
3,P00041,Webcam Full HD,Home & Kitchen,Zenith
4,P00029,T-Shirt,Clothing,KiddoFun
5,P00023,Cookware Set,Books,ReadMore
6,P00030,Dress Shirt,Clothing,UrbanStyle
7,P00028,Jeans,Toys & Games,KiddoFun
8,P00031,Kids Toy Car,Sports & Outdoors,Apex
9,P00001,Wireless Earbuds,Clothing,Apex


### Create dim_time

In [52]:
dim_time = (df_copy[["OrderDate"]]
            .drop_duplicates()
            .sort_values("OrderDate")
            .reset_index(drop=True)
            )
dim_time["DateKey"] = dim_time["OrderDate"].dt.strftime("%Y%m%d").astype(int)
dim_time["Year"] = dim_time["OrderDate"].dt.year
dim_time["Month"] = dim_time["OrderDate"].dt.month
dim_time["MonthName"] = dim_time["OrderDate"].dt.month_name()
dim_time["Quarter"] = dim_time["OrderDate"].dt.quarter
dim_time["Day"] = dim_time["OrderDate"].dt.day

In [53]:
df_copy.head(2)

,OrderID,OrderDate,OrderYear,OrderMonth,CustomerID,CustomerName,ProductID,ProductName,Category,Brand,...,GrossAmount,DiscountedAmount,NetAmount,TotalAmount,PaymentMethod,OrderStatus,City,State,Country,SellerID
0,ORD0000001,2023-01-31,2023,January,CUST001504,Vihaan Sharma,P00014,Drone Mini,Books,BrightLux,...,319.77,0.00,319.77,319.86,Debit Card,Delivered,Washington,DC,India,SELL01967
1,ORD0000002,2023-12-30,2023,December,CUST000178,Pooja Kumar,P00040,Microphone,Home & Kitchen,UrbanStyle,...,251.37,12.57,238.80,259.64,Amazon Pay,Delivered,Fort Worth,TX,United States,SELL01298


### Create fact_sales

In [54]:
fact_sale = df_copy.merge(dim_time[["OrderDate","DateKey"]],
                          on="OrderDate",
                          how="left"
                          )
fact_sale = fact_sale[[
    "OrderID",
    "DateKey",
    "CustomerID",
    "ProductID",
    "SellerID",
    "Quantity",
    "UnitPrice",
    "Discount",
    "Tax",
    "ShippingCost",
    "GrossAmount",
    "DiscountedAmount",
    "NetAmount",
    "TotalAmount",
    "PaymentMethod",
    "OrderStatus"
]]
fact_sale

,OrderID,DateKey,CustomerID,ProductID,SellerID,Quantity,UnitPrice,Discount,Tax,ShippingCost,GrossAmount,DiscountedAmount,NetAmount,TotalAmount,PaymentMethod,OrderStatus
0,ORD0000001,20230131,CUST001504,P00014,SELL01967,3,106.59,0.00,0.00,0.09,319.77,0.00,319.77,319.86,Debit Card,Delivered
1,ORD0000002,20231230,CUST000178,P00040,SELL01298,1,251.37,0.05,19.10,1.74,251.37,12.57,238.80,259.64,Amazon Pay,Delivered
2,ORD0000003,20220510,CUST047516,P00044,SELL00908,3,35.03,0.10,7.57,5.91,105.09,10.51,94.58,108.06,Debit Card,Delivered
3,ORD0000004,20230718,CUST030059,P00041,SELL01164,5,33.58,0.15,11.42,5.53,167.90,25.18,142.72,159.66,Cash on Delivery,Delivered
4,ORD0000005,20230204,CUST048677,P00029,SELL01411,2,515.64,0.25,38.67,9.23,1031.28,257.82,773.46,821.36,Credit Card,Cancelled
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,ORD0099996,20230307,CUST001356,P00047,SELL00041,2,492.34,0.00,78.77,2.75,984.68,0.00,984.68,1066.20,UPI,Delivered
99996,ORD0099997,20211124,CUST031254,P00046,SELL01449,5,449.30,0.00,179.72,6.07,2246.50,0.00,2246.50,2432.29,Credit Card,Delivered
99997,ORD0099998,20230429,CUST012579,P00030,SELL00028,4,232.40,0.00,74.37,12.43,929.60,0.00,929.60,1016.40,Cash on Delivery,Delivered
99998,ORD0099999,20211101,CUST026243,P00046,SELL00324,1,294.05,0.00,23.52,13.09,294.05,0.00,294.05,330.66,Debit Card,Delivered


### Export for SQL

In [55]:
dim_customer.to_csv("dim_customer.csv",index=False)
dim_product.to_csv("dim_product.csv",index=False)
dim_time.to_csv("dim_time.csv",index=False)
fact_sale.to_csv("fact_sale.csv",index=False)